In [1]:
import duckdb
import sys
sys.path.append('..')

from src.utils import make_corpus
from src.semantic import build_semantic_index, semantic_search

# Build corpus

In [2]:
# Read data and drop missing values
c2 = duckdb.connect()
data = c2.execute(f"SELECT * FROM read_parquet('../data/raw/merged.parquet')").df()
data.dropna(subset=['product_title'], inplace=True)

In [3]:
# Extract fields for retrieval
cols = ['title', 'text', 'product_title', 'main_category', 'store']

corpus = make_corpus(df=data, cols=cols, asin="asin")

# Save indices

In [4]:
# BM25 index

In [ ]:
# Semantic index 
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")
semantic_index_path = '../data/processed/embedding.faiss'
build_semantic_index(corpus, model, semantic_index_path)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


# Retrieve results

In [6]:
queries = ["Wet wipes"]

In [7]:
for q in queries:
    print(f"QUERY: {q}\n")

    # print("BM25 top results:")
    # for rank, (product, score) in enumerate(bm25_search(q), start=1):
    #     print(f"{rank}. ({score:.3f}) {product}")

    print("\nSemantic search top results:")
    semantic_results = semantic_search(q, semantic_index_path, model, data.product_title)
    for rank, (product, score) in enumerate(semantic_results, start=1):
        print(f"{rank}. ({score:.3f}) {product}")
    print()

QUERY: Wet wipes


Semantic search top results:
1. (0.889) Audiowipes Towelettes Portable Pouch - by Audiologist's Choice
2. (0.883) Pampers Baby Fresh Water Baby Wipes 3X Pop-Top Packs, 192 Count
3. (0.850) Lens Wipes Pre-moistened Eye Glasses Cleaner Wipes 120 Individually Packaged for Cleaning Glasses Sunglasses Computer Screens Touchscreens Monitors
4. (0.849) Gonioa Baby Wipes Dispenser, Baby Wipes Case, Large Capacity Baby Wipe Holder Keeps Diaper Wipes Fresh, Easy Open & Close Wipe Container with Non-Slip Feet (White & Pink)
5. (0.839) Rinse Free Sponge Bath Wipes (30-pack) | Extra Large & Thick No Rinse Waterless Body Wipe Sponges - Disposable Shower Cleaning Wash Cloths for Kids Adults Surgery Elder Care Gym Travel & More (1 Pack)

